# Telugu preprocessing

Runs `Telugu/preprocess.py` on a Colab CPU runtime with High-RAM, since alignment is CPU work and needs no GPU.
Run all cells and paste a Hugging Face token with write access when asked. Each stage stops the notebook if it
fails.

1. **prepare** selects the 5 h + 1 h male-speaker subset of IndicTTS Telugu and resamples it to 22.05 kHz
2. **align** builds the Telugu lexicon, trains a Montreal Forced Aligner model on the subset and aligns it (about 3 h)
3. **features** extracts mel spectrograms, pitch, energy and phone durations

The results go to the private dataset repo `gsri-18/fs2-iiith-data` as `Telugu_preprocessed.zip`, the training
input, and `Telugu_mfa.zip`, the lexicon, trained aligner and split lists.

In [ ]:
import os
DATA_REPO = "gsri-18/fs2-iiith-data"
print(os.cpu_count(), "CPU cores")

In [ ]:
import getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (write access): ").strip()
from huggingface_hub import whoami
print("logged in as", whoami()["name"])

## Setup

In [ ]:
import re, subprocess, sys, zipfile
from huggingface_hub import hf_hub_download

zipfile.ZipFile(hf_hub_download(DATA_REPO, "code.zip", repo_type="dataset", force_download=True)).extractall("/content/project")
!pip -q install "setuptools<81" pyarrow pyworld tgt g2p_en inflect unidecode librosa soundfile scikit-learn

PY = sys.executable  # Colab's Python, not the one in the aligner's environment


def run(cmd, keep=None, drop=None):
    """Run a shell command, print its filtered output and raise if it fails."""
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        for part in line.replace("\r", "\n").split("\n"):
            if part.strip() and (not keep or re.search(keep, part)) and not (drop and re.search(drop, part)):
                print(part, flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{cmd} failed with exit code {proc.returncode}")

Montreal Forced Aligner gets its own micromamba environment, and only its `mfa` command is used.

In [ ]:
%cd /content
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
!./bin/micromamba create -q -y -p /content/mfa_env -c conda-forge python=3.11 montreal-forced-aligner=3 > /dev/null
os.environ["PATH"] = "/content/mfa_env/bin:" + os.environ["PATH"]
os.environ["MFA_ROOT_DIR"] = "/content/mfa_root"
!mfa version

IndicTTS Telugu, the parquet export of `SPRINGLab/IndicTTS_Telugu` (about 6.4 GB).

In [ ]:
dst = "/content/project/downloads/IndicTTS_Telugu"
os.makedirs(dst, exist_ok=True)
for i in range(4):
    src = hf_hub_download("SPRINGLab/IndicTTS_Telugu", f"default/train/000{i}.parquet",
                          repo_type="dataset", revision="refs/convert/parquet")
    if not os.path.exists(f"{dst}/000{i}.parquet"):
        os.symlink(src, f"{dst}/000{i}.parquet")
print(sorted(os.listdir(dst)))

## Prepare

In [ ]:
%cd /content/project/Telugu
run(f"{PY} -W ignore preprocess.py config.yaml --stage prepare", drop=r"it/s|s/it")
n = len([f for f in os.listdir("raw_data/te_male") if f.endswith(".wav")])
print(n, "utterances written")
assert n > 3000

## Align

In [ ]:
run(f"{PY} -W ignore preprocess.py config.yaml --stage align --num_jobs {os.cpu_count()}",
    keep=r"INFO|Iteration|rror|Traceback|lexicon|Done")
n = int(subprocess.check_output("find preprocessed_data/TextGrid -name '*.TextGrid' | wc -l", shell=True))
print(n, "TextGrids")
assert n > 3000

## Features

In [ ]:
run(f"{PY} -W ignore preprocess.py config.yaml --stage features", drop=r"it/s|s/it")
!cat preprocessed_data/stats.json
!wc -l preprocessed_data/train.txt preprocessed_data/val.txt

## Upload

In [ ]:
from huggingface_hub import HfApi

%cd /content/project
with zipfile.ZipFile("/content/Telugu_preprocessed.zip", "w", zipfile.ZIP_STORED) as z:
    for d, _, files in os.walk("Telugu/preprocessed_data"):
        for f in files:
            z.write(os.path.join(d, f))
with zipfile.ZipFile("/content/Telugu_mfa.zip", "w") as z:
    for f in ["Telugu/mfa/telugu_lexicon.txt", "Telugu/mfa/telugu_mfa_acoustic.zip",
              "Telugu/raw_data/split.json", "Telugu/raw_data/subset_stats.json"]:
        z.write(f)

assert os.path.getsize("/content/Telugu_preprocessed.zip") > 100e6, "features are missing"
api = HfApi()
for name in ["Telugu_preprocessed.zip", "Telugu_mfa.zip"]:
    print(name, round(os.path.getsize(f"/content/{name}") / 1e6, 1), "MB")
    api.upload_file(path_or_fileobj=f"/content/{name}", path_in_repo=name, repo_id=DATA_REPO, repo_type="dataset")
print("uploaded")